# Indian TTS — Staged Training with Validation Gates

Train a custom Indian English TTS model (male & female voices) step by step.

**How this works:** Instead of training for 8 hours and hoping it works, we train in stages:

| Stage | Time | What It Checks |
|-------|------|----------------|
| **0 — Sanity Check** | ~2 min | Does it even run without crashing? |
| **1 �� Smoke Test** | ~30 min | Are losses going down? |
| **2 — Early Signal** | ~1 hour | Is audio structure emerging? |
| **3 — Quality Gate** | ~2 hours | Do male/female sound different? |
| **4 — Full Training** | ~6-8 hours | Is speech becoming intelligible? |
| **5 — Extended** | ~24 hours | Good quality Indian English? |

Each stage has **PASS/FAIL checks**. Only move to the next stage if the current one passes.
Each stage produces **audio samples** you can listen to.

**Requirements:** Google Colab Pro+ with A100 GPU

---
## Setup (run once)

In [ ]:
# Check GPU — MUST be A100
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
assert torch.cuda.is_available(), "No GPU! Go to Runtime > Change runtime type > A100"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.0f} GB")

In [ ]:
# Clone repo and install
!git clone https://github.com/seetha0712/text2speech_1.git /content/indian_tts 2>/dev/null || echo "Already cloned"
%cd /content/indian_tts
!git checkout claude/custom-indian-tts-model-TUAjJ
!git pull origin claude/custom-indian-tts-model-TUAjJ

!pip install -q -r requirements.txt
!pip install -q -e .
!apt-get install -qq espeak-ng > /dev/null 2>&1
print("\nSetup complete!")

In [ ]:
# Login to HuggingFace (needed for Common Voice download)
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# Download Indian English data (CC-0 + CC-BY 4.0 — legally safe)
# This takes ~10-20 min depending on internet speed
!python -m indian_tts.data.preprocess \
    --source all \
    --output /content/data \
    --max-hours 15 \
    --min-upvotes 2

In [ ]:
# Create config pointing to our data
import yaml

with open('configs/colab_a100_config.yaml') as f:
    config = yaml.safe_load(f)

config['data']['training_files'] = '/content/data/train.txt'
config['data']['validation_files'] = '/content/data/val.txt'
config['paths']['output_dir'] = '/content/outputs'
config['paths']['checkpoint_dir'] = '/content/outputs/checkpoints'
config['paths']['log_dir'] = '/content/outputs/logs'

with open('/content/stage_config.yaml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print("Config ready!")
print(f"Batch size: {config['training']['batch_size']}")

# Verify data exists
for f in ['train.txt', 'val.txt']:
    path = f'/content/data/{f}'
    import os
    if os.path.exists(path):
        with open(path) as fh:
            lines = [l for l in fh if l.strip() and not l.startswith('#')]
        print(f"{f}: {len(lines)} samples")
    else:
        print(f"ERROR: {path} not found!")

---
## Stage 0 — Sanity Check (~2 minutes)

**What it checks:** Does the data load? Does the model run? Do we get finite losses?

**If it FAILS:** Something is fundamentally broken (bad data, install issue, etc.)

In [ ]:
!python -m indian_tts.validate --config /content/stage_config.yaml --stage 0

In [ ]:
# Listen to Stage 0 output (will be noise — that's expected!)
import IPython.display as ipd
import os

sample_dir = '/content/outputs/stage_0_samples'
if os.path.exists(sample_dir):
    for f in sorted(os.listdir(sample_dir)):
        if f.endswith('.wav'):
            print(f"\n{f} (expect: noise/static — that's OK at stage 0)")
            ipd.display(ipd.Audio(os.path.join(sample_dir, f)))
else:
    print("No samples found. Stage 0 may have failed.")

### Stage 0 Decision
- **PASSED?** Proceed to Stage 1 below.
- **FAILED?** Check the error messages. Common fixes:
  - OOM error → Reduce `batch_size` to 32 in the config cell above
  - Data not found → Re-run the data download cell
  - Import error → Re-run the install cell

---
## Stage 1 — Smoke Test (~30 minutes)

**What it checks:** Are losses going down? Are gradients flowing? Is audio non-silent?

**If it FAILS:** Learning rate or model config may need adjustment.

In [ ]:
# Find latest checkpoint from Stage 0
import glob
ckpts = sorted(glob.glob('/content/outputs/checkpoints/checkpoint_*.pt'))
resume = ckpts[-1] if ckpts else None
print(f"Resuming from: {resume}")

!python -m indian_tts.validate \
    --config /content/stage_config.yaml \
    --stage 1 \
    --resume {resume if resume else ''}

In [ ]:
# Listen to Stage 1 output (expect: buzzy/noisy but with SOME structure)
sample_dir = '/content/outputs/stage_1_samples'
if os.path.exists(sample_dir):
    for f in sorted(os.listdir(sample_dir)):
        if f.endswith('.wav'):
            print(f"\n{f} (expect: noisy/buzzy but not pure static)")
            ipd.display(ipd.Audio(os.path.join(sample_dir, f)))
else:
    print("No samples found.")

### Stage 1 Decision
- **PASSED + audio has some energy?** Proceed to Stage 2.
- **FAILED?** Training is unstable. Try:
  - Lower learning rate: change `learning_rate` to `0.0001` in config
  - Reduce batch size to 32

---
## Stage 2 — Early Signal (~1 hour total)

**What it checks:** Audio structure emerging? Male/female sound different?

**If it FAILS:** Model may need more data or architecture tweaks.

In [ ]:
ckpts = sorted(glob.glob('/content/outputs/checkpoints/checkpoint_*.pt'))
resume = ckpts[-1] if ckpts else None
print(f"Resuming from: {resume}")

!python -m indian_tts.validate \
    --config /content/stage_config.yaml \
    --stage 2 \
    --resume {resume if resume else ''}

In [ ]:
# Listen to Stage 2 output (expect: speech-like sounds, maybe garbled)
sample_dir = '/content/outputs/stage_2_samples'
if os.path.exists(sample_dir):
    for f in sorted(os.listdir(sample_dir)):
        if f.endswith('.wav'):
            print(f"\n{f} (expect: speech-like noise, not clear words yet)")
            ipd.display(ipd.Audio(os.path.join(sample_dir, f)))
else:
    print("No samples found.")

### Stage 2 Decision
- **PASSED + male/female sound different?** Great progress! Proceed to Stage 3.
- **FAILED?** Model may need more data. Consider increasing `--max-hours` in data download.

---
## Stage 3 — Quality Gate (~2 hours total)

**What it checks:** Vowel-like sounds? Different texts produce different outputs?

**This is the key decision point.** If Stage 3 passes, the model is learning properly and investing 6+ more hours will yield results.

In [ ]:
ckpts = sorted(glob.glob('/content/outputs/checkpoints/checkpoint_*.pt'))
resume = ckpts[-1] if ckpts else None
print(f"Resuming from: {resume}")

!python -m indian_tts.validate \
    --config /content/stage_config.yaml \
    --stage 3 \
    --resume {resume if resume else ''}

In [ ]:
# Listen to Stage 3 output (expect: vowel sounds, speech-like patterns)
sample_dir = '/content/outputs/stage_3_samples'
if os.path.exists(sample_dir):
    for f in sorted(os.listdir(sample_dir)):
        if f.endswith('.wav'):
            print(f"\n{f} (expect: speech-like, maybe a few recognizable sounds)")
            ipd.display(ipd.Audio(os.path.join(sample_dir, f)))
else:
    print("No samples found.")

### Stage 3 Decision — THE GO/NO-GO POINT

**If PASSED:** The model is learning correctly. Proceed to Stage 4 (6-8 hours). Worth the investment.

**If FAILED:** STOP. Do NOT spend 6 more hours. Investigate:
- Is the data too noisy? Try increasing `--min-upvotes 3` in data download
- Not enough data? Increase `--max-hours 25`
- Model too small? This is unlikely with VITS2 defaults

---

## Stage 4 — Full Training (~6-8 hours)

**What it checks:** Partially recognizable words? Consistent speaker identity?

**IMPORTANT:** Save to Google Drive before starting (in case Colab disconnects).

In [ ]:
# Mount Google Drive for checkpoint backup
from google.colab import drive
drive.mount('/content/drive')

# Auto-backup checkpoints to Drive every time they're saved
import shutil
drive_backup = '/content/drive/MyDrive/indian_tts_checkpoints'
os.makedirs(drive_backup, exist_ok=True)

# Copy current checkpoints to Drive
ckpts = sorted(glob.glob('/content/outputs/checkpoints/checkpoint_*.pt'))
for ckpt in ckpts[-2:]:  # Copy last 2
    dest = os.path.join(drive_backup, os.path.basename(ckpt))
    if not os.path.exists(dest):
        shutil.copy2(ckpt, dest)
        print(f"Backed up: {os.path.basename(ckpt)}")

# Also save config
shutil.copy2('/content/stage_config.yaml', drive_backup)
print(f"\nBackups in: {drive_backup}")

In [ ]:
# Start TensorBoard (optional — monitor in real time)
%load_ext tensorboard
%tensorboard --logdir /content/outputs/logs

In [ ]:
ckpts = sorted(glob.glob('/content/outputs/checkpoints/checkpoint_*.pt'))
resume = ckpts[-1] if ckpts else None
print(f"Resuming from: {resume}")
print("This will take ~6-8 hours...")

!python -m indian_tts.validate \
    --config /content/stage_config.yaml \
    --stage 4 \
    --resume {resume if resume else ''}

In [ ]:
# Listen to Stage 4 output (expect: partially intelligible speech!)
sample_dir = '/content/outputs/stage_4_samples'
if os.path.exists(sample_dir):
    for f in sorted(os.listdir(sample_dir)):
        if f.endswith('.wav'):
            print(f"\n{f}")
            ipd.display(ipd.Audio(os.path.join(sample_dir, f)))
else:
    print("No samples found.")

# Backup to Drive
ckpts = sorted(glob.glob('/content/outputs/checkpoints/checkpoint_*.pt'))
for ckpt in ckpts[-2:]:
    dest = os.path.join(drive_backup, os.path.basename(ckpt))
    if not os.path.exists(dest):
        shutil.copy2(ckpt, dest)
        print(f"Backed up: {os.path.basename(ckpt)}")

---
## Stage 5 — Extended Training (~24 hours, optional)

Only run this if Stage 4 showed promising results and you want higher quality.
You may need to run this across multiple Colab sessions (resume from Drive backup).

In [ ]:
# Resume from Drive if this is a new Colab session
import glob, os

drive_backup = '/content/drive/MyDrive/indian_tts_checkpoints'
local_ckpts = sorted(glob.glob('/content/outputs/checkpoints/checkpoint_*.pt'))
drive_ckpts = sorted(glob.glob(f'{drive_backup}/checkpoint_*.pt'))

if not local_ckpts and drive_ckpts:
    # Restore from Drive
    import shutil
    os.makedirs('/content/outputs/checkpoints', exist_ok=True)
    latest = drive_ckpts[-1]
    shutil.copy2(latest, '/content/outputs/checkpoints/')
    print(f"Restored from Drive: {os.path.basename(latest)}")
    resume = f'/content/outputs/checkpoints/{os.path.basename(latest)}'
else:
    resume = local_ckpts[-1] if local_ckpts else None

print(f"Resuming from: {resume}")

In [ ]:
!python -m indian_tts.validate \
    --config /content/stage_config.yaml \
    --stage 5 \
    --resume {resume if resume else ''}

---
## Generate Speech (after any successful stage)

You can generate speech from any checkpoint, not just the final one.

In [ ]:
from indian_tts.inference import IndianTTS
import IPython.display as ipd
import glob

# Load latest checkpoint
ckpts = sorted(glob.glob('/content/outputs/checkpoints/checkpoint_*.pt'))
tts = IndianTTS(ckpts[-1])

# Your custom texts
texts = [
    "Hello, welcome to our Indian text to speech system.",
    "The quarterly results show a fifteen percent increase in revenue.",
    "Please proceed to gate number seven for boarding.",
]

for text in texts:
    print(f"\n--- {text} ---")
    for voice in ['male', 'female']:
        audio = tts.synthesize(text, voice=voice)
        print(f"[{voice.upper()}]")
        ipd.display(ipd.Audio(audio, rate=tts.sampling_rate))